# Description

To load this from remote:
```
from transformers import pipeline

# It will download once and cache it for future use
ner_pipe = pipeline("ner", model="alexd063/bio-clinicalbert-finetuned-medmentions")
results = ner_pipe("Patient shows symptoms of acute appendicitis.")
print(results)
```

In [1]:
%%capture
!pip install transformers datasets evaluate seqeval accelerate torch numpy
!pip install --upgrade transformers

In [2]:
import os
# Suppress all tqdm/HuggingFace progress bars before any library is imported
os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"

In [3]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback
)
from huggingface_hub import notebook_login, whoami

In [4]:
import os
os.environ["TQDM_DISABLE"] = "1"

from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from datasets import disable_progress_bars
disable_progress_bars()

from huggingface_hub.utils import disable_progress_bars as hub_disable_progress_bars
hub_disable_progress_bars()

In [5]:
PUSH_TO_HUB = True  # Set to False to skip login and hub push

if PUSH_TO_HUB:
    notebook_login()


In [6]:
class Config:
    MODEL_ID = "emilyalsentzer/Bio_ClinicalBERT"
    DATASET_ID = "Ben10x/MedMentions-MTI881-NER"
    HUB_REPO_ID = "alexd063/bio-clinicalbert-finetuned-medmentions"

    MAX_LENGTH = 512
    BATCH_SIZE = 16
    EPOCHS = 20
    LEARNING_RATE = 2e-5
    WARMUP_RATIO = 0.10
    WEIGHT_DECAY = 0.01

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = Config()


In [7]:
import time
notebook_start_time = time.time()


In [13]:
# ==========================================
# 1. Load Dataset & Extract Labels
# ==========================================
dataset = load_dataset("Ben10x/MedMentions-MTI881-NER")

# the dataset stores tags as strings (e.g., 'B-T062', 'I-T062', 'O').
# we need to extract all unique tags to create our label mappings.
unique_tags = set()
for split in dataset.keys():
    for row in dataset[split]["ner_tags"]:
        unique_tags.update(row)

label_list = sorted(list(unique_tags))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

# ==========================================
# 2. Tokenization & Label Alignment
# ==========================================
# tokenization
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_ID)

def tokenize_and_align_labels(examples):
    # tokenize the pre-split words
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=config.MAX_LENGTH
    )

    labels = []
    for i, label_sequence in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # special tokens (like [CLS] and [SEP]) map to None. we set their label to -100 so they are ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # only label the first token of a given word.
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label_sequence[word_idx]])
            # for subsequent tokens of the same word, also assign -100 (or you can assign the I- tag).
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# apply to train, validation, and test splits
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)


In [14]:
%%capture
# capture widget output, github cannot show widgets.

# ==========================================
# 3. Metrics
# ==========================================

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": np.round(results["overall_precision"], 3),
        "recall": np.round(results["overall_recall"], 3),
        "f1": np.round(results["overall_f1"], 3),
        "accuracy": np.round(results["overall_accuracy"], 3),
    }

In [ ]:

# ==========================================
# 4. Training
# ==========================================

training_args = TrainingArguments(
    output_dir="./bio-clinicalbert-medmentions",
    learning_rate=config.LEARNING_RATE,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE,
    num_train_epochs=config.EPOCHS,
    weight_decay=config.WEIGHT_DECAY,
    warmup_ratio=config.WARMUP_RATIO,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    hub_model_id=config.HUB_REPO_ID if PUSH_TO_HUB else None
)

model = AutoModelForTokenClassification.from_pretrained(
    config.MODEL_ID,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

# save the final model locally
trainer.save_model("./bio-clinicalbert-medmentions-final")


In [10]:
# ==========================================
# 5. Assessment on the Test Set
# ==========================================

test_results = trainer.evaluate(tokenized_datasets["test"])

print("--- Test Set Evaluation Results ---")
for key, value in test_results.items():
    # formatting key names for better readability (e.g., eval_f1 -> f1)
    metric_name = key.replace("eval_", "")
    print(f"{metric_name:10}: {value:.4f}")

# specific examples
# predictions, labels, metrics = trainer.predict(tokenized_datasets["test"])
# print("\nDetailed Test Metrics:", metrics)


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': '0.452', 'eval_precision': '0.612', 'eval_recall': '0.647', 'eval_f1': '0.629', 'eval_accuracy': '0.88', 'eval_runtime': '5.733', 'eval_samples_per_second': '510.4', 'eval_steps_per_second': '31.92', 'epoch': '8'}
--- Test Set Evaluation Results ---
loss      : 0.4520
precision : 0.6120
recall    : 0.6470
f1        : 0.6290
accuracy  : 0.8800
runtime   : 5.7327
samples_per_second: 510.4030
steps_per_second: 31.9220
epoch     : 8.0000


In [ ]:
# ==========================================
# 6. Training Curves
# ==========================================
import matplotlib.pyplot as plt

history = trainer.state.log_history

train_loss = [x['loss'] for x in history if 'loss' in x and 'eval_loss' not in x]
train_steps = [x['step'] for x in history if 'loss' in x and 'eval_loss' not in x]

eval_f1 = [x['eval_f1'] for x in history if 'eval_f1' in x]
eval_epochs = [x['epoch'] for x in history if 'eval_f1' in x]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(train_steps, train_loss, color='steelblue', linewidth=1.5, label='Train Loss')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss (Baseline Bio_ClinicalBERT NER)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

if eval_f1:
    axes[1].plot(eval_epochs, eval_f1, color='darkorange', marker='o', linewidth=1.5, label='Eval F1')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1')
    axes[1].set_title('Validation F1 per Epoch (Baseline)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ner_training_curves_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ner_training_curves_baseline.png")

In [ ]:
# ==========================================
# 7. Per-Entity-Type Analysis on Test Set
# ==========================================
import seaborn as sns
from seqeval.metrics import classification_report as seqeval_report

# Run prediction on test set
test_predictions, test_labels_raw, _ = trainer.predict(tokenized_datasets["test"])
test_predictions_ids = np.argmax(test_predictions, axis=2)

# Convert ids to label strings, filter out -100
true_labels = []
true_predictions = []
for pred_row, label_row in zip(test_predictions_ids, test_labels_raw):
    true_pred = []
    true_lab = []
    for p, l in zip(pred_row, label_row):
        if l != -100:
            true_pred.append(id2label[p])
            true_lab.append(id2label[l])
    true_predictions.append(true_pred)
    true_labels.append(true_lab)

# Print formatted classification report
print("=" * 60)
print("Per-Entity Classification Report (Test Set)")
print("=" * 60)
print(seqeval_report(true_labels, true_predictions))

# Build per-entity F1 bar chart
report_dict = seqeval_report(true_labels, true_predictions, output_dict=True)
rows = []
for entity, metrics in report_dict.items():
    if isinstance(metrics, dict) and entity not in ('micro avg', 'macro avg', 'weighted avg'):
        rows.append({'entity': entity, 'f1': metrics['f1-score'], 'support': metrics['support']})

df_report = pd.DataFrame(rows).sort_values('f1', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(df_report) * 0.3)))
colors = ['#d62728' if f < 0.3 else '#ff7f0e' if f < 0.6 else '#2ca02c' for f in df_report['f1']]
ax.barh(df_report['entity'], df_report['f1'], color=colors)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='F1=0.5 threshold')
ax.set_xlabel('F1 Score')
ax.set_title('Per-Entity-Type F1 Score — Baseline Bio_ClinicalBERT NER (Test Set)')
ax.legend()
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('ner_per_entity_f1_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ner_per_entity_f1_baseline.png")

In [ ]:
# ==========================================
# 8. Sample Predictions (Test Set)
# ==========================================
import random

# Show 5 examples: the first test sentence with actual entities
shown = 0
target = 5

for i, (preds, labs, tokens) in enumerate(zip(true_predictions, true_labels, tokenized_datasets["test"]["tokens"] if "tokens" in tokenized_datasets["test"].features else [None]*len(true_predictions))):
    if shown >= target:
        break
    # Only show sentences that have at least one non-O entity
    if not any(l != 'O' for l in labs):
        continue
    print(f"\n--- Example {shown+1} (Test sample #{i}) ---")
    print(f"{'Token':<30} {'True':>15} {'Predicted':>15} {'Match':>6}")
    print("-" * 70)
    for p, l in zip(preds, labs):
        match = "✓" if p == l else "✗"
        print(f"{'[token]':<30} {l:>15} {p:>15} {match:>6}")
    shown += 1

# More informative version using word-level tokens from the raw dataset
print("\n\n--- Detailed: Word | True | Predicted ---")
raw_dataset = load_dataset(config.DATASET_ID)
shown = 0
for i, example in enumerate(raw_dataset["test"]):
    if shown >= 5:
        break
    tokens = example["tokens"]
    labs = true_labels[i] if i < len(true_labels) else []
    preds = true_predictions[i] if i < len(true_predictions) else []
    if not any(l != 'O' for l in labs):
        continue
    print(f"\n--- Example {shown+1} ---")
    print(f"{'Word':<25} {'True':>15} {'Predicted':>15}")
    print("-" * 57)
    for w, l, p in zip(tokens, labs, preds):
        marker = " <<<" if l != p else ""
        print(f"{w:<25} {l:>15} {p:>15}{marker}")
    shown += 1

In [11]:
notebook_end_time = time.time()
elapsed = notebook_end_time - notebook_start_time
hours, rem = divmod(elapsed, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Total notebook execution time: {int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}")


Total notebook execution time: 00:10:57


In [12]:

# ==========================================
# 7. Push to Hugging Face Hub
# ==========================================

if PUSH_TO_HUB:
    trainer.push_to_hub(
        commit_message="End of training",
        finetuned_from=config.MODEL_ID,
        dataset=config.DATASET_ID
    )
    tokenizer.push_to_hub(repo_id=config.HUB_REPO_ID)

    # sanity check: run inference from the remote model
    from transformers import pipeline

    ner_pipe = pipeline("ner", model=config.HUB_REPO_ID)
    results = ner_pipe("Patient shows symptoms of acute appendicitis.")
    print(results)


No files have been modified since last commit. Skipping to prevent empty commit.


[{'entity': 'B-T033', 'score': np.float32(0.9926732), 'index': 3, 'word': 'symptoms', 'start': 14, 'end': 22}, {'entity': 'B-T038', 'score': np.float32(0.99708194), 'index': 5, 'word': 'acute', 'start': 26, 'end': 31}, {'entity': 'I-T038', 'score': np.float32(0.99832183), 'index': 6, 'word': 'app', 'start': 32, 'end': 35}, {'entity': 'I-T038', 'score': np.float32(0.99675447), 'index': 7, 'word': '##end', 'start': 35, 'end': 38}, {'entity': 'I-T038', 'score': np.float32(0.99442), 'index': 8, 'word': '##icit', 'start': 38, 'end': 42}, {'entity': 'I-T038', 'score': np.float32(0.99552333), 'index': 9, 'word': '##is', 'start': 42, 'end': 44}]
